In [1]:
import pandas as pd
import os
from landcover_assignment.transition_matrix import TransitionMatrix
from landcover_assignment.landcover import LandCover
from landcover_assignment.afforestation import Afforestation

# Landcover Assignment Example

The landcover assignment module takes scenario inputs and inputs generated from the [grassland_production](https://colmduff.github.io/grassland_production/html/index.html) module and distributes the spared area to various land uses. It considers soil type in the process (avaialbility of mineral or organic soils). It also considers soil groups in relation to the availability of organic soils and the forest yield class. 

It produces the land use change dataframe for current and future land uses, as well as a transition matrix, and an afforestation dataframe that will then be used by the [cbm_runner](https://colmduff.github.io/cbm_runner/html/index.html) module

## Set UP

In [2]:
input_path = "../tests/data/"

baseline = 2020 #the calibration year
target_year = 2050 #the target year

scenario_dataframe = pd.read_csv(os.path.join(input_path, "scenario_dataframe.csv"), index_col=0) #scenario inputs 

spared_area = pd.read_csv(os.path.join(input_path, "spared_area.csv"), index_col=0) #total spared area 
spared_area.columns = spared_area.columns.astype(int) #year columns as int
spared_area_breakdown = pd.read_csv(os.path.join(input_path, "spared_area_breakdown.csv"), index_col=0) #spared area by soil group
grassland_area = pd.read_csv(os.path.join(input_path, "total_grassland_area.csv"), index_col=0) #total grassland area remaining
grassland_area.columns = grassland_area.columns.astype(int)


## Classes 

The ```TransitionMatrix``` class wil produce the transition matrix. While the ```LandCover``` class produces the land use change dataframe, including the proportions of various soil types and other environmental factors. 

Finally, the ```Afforestation``` class produces the afforested area by yield class for use in the ```cbm_runner``` module. 

In [3]:
transition = TransitionMatrix(
    baseline, target_year, scenario_dataframe, grassland_area, spared_area, spared_area_breakdown
)

land = LandCover(
    baseline, target_year, scenario_dataframe, grassland_area, spared_area, spared_area_breakdown
)

affor = Afforestation(baseline, target_year, scenario_dataframe, transition.create_transition_matrix())

### Combined land use dataframe 

In [4]:
print(land.combined_future_land_use_area())

    farm_id  year            land_use       area_ha  share_mineral  \
0     -2020  2020           grassland  4.218508e+06       0.919424   
1     -2020  2020             wetland  1.212500e+06       0.000047   
2     -2020  2020            cropland  7.476981e+05       1.000000   
3     -2020  2020              forest  7.300988e+05       0.472948   
4     -2020  2020          settlement  1.261121e+05       0.984141   
5     -2020  2020  farmable_condition  0.000000e+00       1.000000   
6         0  2050             wetland  1.212500e+06       0.000047   
7         0  2050              forest  2.571770e+06       0.850375   
8         0  2050            cropland  7.476981e+05       1.000000   
9         0  2050  farmable_condition  1.105181e+06       1.000000   
10        0  2050           grassland  1.413048e+06       0.759448   
11        0  2050          settlement  1.261121e+05       0.984141   
12        1  2050             wetland  1.212500e+06       0.000047   
13        1  2050   

### Transition Matrix

In [5]:
print(transition.create_transition_matrix())

       Grassland_to_Grassland  Grassland_to_Wetland  Grassland_to_Cropland  \
-2020            0.000000e+00                   0.0                    0.0   
 0              -2.805460e+06                   0.0                    0.0   
 1              -2.720430e+06                   0.0                    0.0   
 2              -2.720430e+06                   0.0                    0.0   
 3              -3.590237e+06                   0.0                    0.0   

       Grassland_to_Forest  Grassland_to_Settlement  \
-2020         0.000000e+00                      0.0   
 0            1.700279e+06                      0.0   
 1            2.704498e+06                      0.0   
 2            0.000000e+00                      0.0   
 3            0.000000e+00                      0.0   

       Grassland_to_Farmable_Condition  Wetland_to_Grassland  \
-2020                     0.000000e+00                   0.0   
 0                        1.105181e+06                   0.0   
 1      

### Afforestation Dataframe

In [6]:
print(affor.gen_cbm_afforestation_dataframe(spared_area_breakdown))

    scenario species yield_class     total_area
0         -1   Sitka     YC17_20       0.000000
1         -1   Sitka     YC20_24       0.000000
2         -1   Sitka     YC24_30       0.000000
3         -1     SGB         YC6       0.000000
4         -1     SGB         YC6       0.000000
5         -1     SGB         YC6       0.000000
6          0   Sitka     YC24_30  793471.392979
7          0     SGB         YC6  340059.168420
8          0   Sitka     YC20_24  396723.679300
9          0     SGB         YC6  170024.433986
10         0   Sitka     YC17_20       0.000000
11         0     SGB         YC6       0.000000
12         1   Sitka     YC24_30  795908.062041
13         1     SGB         YC6  341103.455160
14         1   Sitka     YC20_24  965626.893014
15         1     SGB         YC6  413840.097006
16         1   Sitka     YC17_20  131613.591886
17         1     SGB         YC6   56405.825094
18         2   Sitka     YC24_30       0.000000
19         2     SGB         YC6       0